In [11]:
import os
import pickle
import numpy as np
import pandas as pd
from collections import defaultdict
import plotly.io as pio
pio.renderers.default = "notebook_connected"
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
from hdbscan import HDBSCAN
import umap
import warnings
from itertools import chain
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from sentence_transformers import SentenceTransformer
from pathlib import Path

In [2]:
embeddings_path = "../data/embeddings"

data_path = "../data/processed"
file_name = 'dataset_final_phrases.pkl'
with open(os.path.join(data_path, file_name), 'rb') as f:
    df = pickle.load(f)

### BERTopic

In [39]:
sentences = df["sentence"].tolist()

file_name_mutli = 'emb_sbert_multi_phrases.pkl'
with open(os.path.join(embeddings_path, file_name_mutli), 'rb') as f:
    embedding_multi = pickle.load(f)

file_name_fr = 'emb_sbert_fr_phrases.pkl'
with open(os.path.join(embeddings_path, file_name_fr), 'rb') as f:
    embedding_fr = pickle.load(f)

file_name_perf = 'emb_sbert_multi2_phrases.pkl'
with open(os.path.join(embeddings_path, file_name_fr), 'rb') as f:
    embedding_multi2 = pickle.load(f)

In [40]:
print(len(sentences))
print(embedding_multi.shape)
print(embedding_fr.shape)
print(embedding_multi2.shape)

37561
(37561, 384)
(37561, 768)
(37561, 768)


In [41]:
warnings.filterwarnings("ignore", category=UserWarning)

product_words = ['montre', 'montres', 'boucle', 'boucles', 'oreilles', 'oreille', 'paire', 'paires', 'lunettes', 'bracelet', 'bracelets', 'collier', 'iphone', 'téléphone', 'pendentif', 'robot', 'robots', 'aspirateur', 'baskets', 'basket', 'chaussure', 'chaussures', 'sandales', 'plantes', 'plante', 'arbres', 'arbre', 'bulbes', 'willemse', 'sommiers', 'jardin', 'bague', 'lampe', 'lampes', 'abat-jour', 'abat jour', 'lampadaire', 'parfum', 'shampoing', 'shampooing', 'shampooings', 'cheveux', 'masque', 'masques', 'crème', 'élastiques', 'manteau', 'bougie', 'cadre', 'écouteurs', 'vélo', 'robe', 'vêtements', 'bijoux', 'sac', 'portable', 'clio', 'luminaire', 'oreillette', 'induction', 'écouteur', 'couette', 'samsung', 'téléphones', 'smartcase', 'abat', 'apple', 'watch', 'shirt', 'tee', 'chemise', 'shirts', 'hortensias', 'orchidée', 'sacs', 'plant', 'reconditionné']

models = {}

for embedding, name in zip([embedding_fr, embedding_multi, embedding_multi2], ["français", "multilingue", "multilingue_2"]):
    print(name)

    vectorizer_model = CountVectorizer(
        stop_words=stopwords.words("french") + product_words,
        ngram_range=(1, 3),
        max_df=0.95
    )

    hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    min_samples=3
    )
    
    topic_model = BERTopic(
        language="french",
        vectorizer_model=vectorizer_model,
        hdbscan_model=hdbscan_model,
        verbose=False,
        nr_topics="auto"
    )
    
    topics, probs = topic_model.fit_transform(sentences, embedding)
    
    topics = np.array(topics)
    
    # Identifier les gros clusters
    topic_sizes = pd.Series(topics).value_counts()
    large_topics = topic_sizes[topic_sizes > 2000].index  # seuil à ajuster selon dataset
    
    for t in large_topics:
        print("Topic :", t)
        # Récupérer les indices des documents dans le gros cluster
        idx = np.where(topics == t)[0]
        embeddings_subset = embedding[idx]
        
        # Diviser le gros cluster avec KMeans
        n_subclusters = int(len(idx) / 200)  # ~200 docs par sous-cluster
        print("Nombre de clusters créés par k-means :", n_subclusters)
        
        emb_norm = normalize(embeddings_subset)
        kmeans = MiniBatchKMeans(
            n_clusters=n_subclusters,
            batch_size=512,
            max_iter=200,
            n_init="auto"
        )
        sub_labels = kmeans.fit_predict(emb_norm)
        
        # Réassigner les labels dans `topics`
        max_topic_id = topics.max() + 1
        for i, doc_idx in enumerate(idx):
            topics[doc_idx] = max_topic_id + sub_labels[i]

        topic_model.update_topics(
            docs=sentences,
            topics=topics,
            vectorizer_model=topic_model.vectorizer_model,
            top_n_words=15
        )

    models[name] = topic_model

français
Topic : -1
Nombre de clusters créés par k-means : 107


2026-02-09 16:53:45,796 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Topic : 0
Nombre de clusters créés par k-means : 78


2026-02-09 16:53:51,034 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


multilingue
Topic : -1
Nombre de clusters créés par k-means : 77


2026-02-09 16:54:11,582 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


multilingue_2
Topic : -1
Nombre de clusters créés par k-means : 113


2026-02-09 16:54:35,527 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Topic : 0
Nombre de clusters créés par k-means : 43


2026-02-09 16:54:39,715 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


### Evaluation

#### Diversité

In [6]:
def topic_diversity(topic_model, top_n=10):
    """
    Calcule la diversité des topics d'un modèle BERTopic.
    """
    topics = topic_model.get_topics()

    # On extrait les mots uniquement (sans les scores)
    topic_words = []
    for topic_id, word_scores in topics.items():
        # BERTopic place les topics -1 et autres meta-topics, donc on ignore topic -1
        if topic_id == -1:
            continue
        top_words = [w for (w, score) in word_scores[:top_n]]
        topic_words.append(top_words)

    # Liste aplatie
    all_words = list(chain.from_iterable(topic_words))
    unique_words = set(all_words)

    return len(unique_words) / len(all_words)


# Calcul du score pour chaque modèle
diversity_scores = {}

for name, model in models.items():
    score = topic_diversity(model, top_n=10)
    diversity_scores[name] = score

diversity_scores

{'français': 0.4954314720812183,
 'multilingue': 0.7647509578544062,
 'multilingue_2': 0.6402298850574712}

#### Score de cohérence

In [7]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

for name, model in models.items():
    topics = model.topics_
    docs_by_topic = defaultdict(list)
    for doc, t in zip(sentences, topics):
        if t != -1:
            docs_by_topic[t].append(doc)
    
    # Générer top words manuellement
    topic_words = []
    for t, docs in docs_by_topic.items():
        vec = TfidfVectorizer(stop_words=stopwords.words("french") + product_words, ngram_range=(1,3))
        X = vec.fit_transform(docs)
        feature_names = np.array(vec.get_feature_names_out())
        # tfidf_sum = X.toarray().sum(axis=0)
        tfidf_sum = np.asarray(X.sum(axis=0)).ravel()
        top_words = feature_names[np.argsort(tfidf_sum)[::-1]][:10].tolist()
        topic_words.append(top_words)
    
    # Tokenisation des documents
    tokenized_docs = [doc.lower().split() for doc in sentences]
    dictionary = Dictionary(tokenized_docs)
    
    # Calcul du score de cohérence
    cm = CoherenceModel(
        topics=topic_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    score = cm.get_coherence()
    print(name, score)

français 0.5427444524333772
multilingue 0.5127186875553322
multilingue_2 0.5400747665314202


#### Embedding-based coherence score

In [8]:
def embedding_coherence(topic_model, embedder, top_n=10):
    """
    Calcule la cohérence basée sur les embeddings pour un modèle BERTopic.
    embedder : modèle sentence-transformers pour transformer les mots en vecteurs.
    """
    topics = topic_model.get_topics()
    scores = []

    for topic_id, word_scores in topics.items():
        if topic_id == -1:
            continue
        top_words = [w for w, _ in word_scores[:top_n]]
        word_embeddings = embedder.encode(top_words)
        sim_matrix = cosine_similarity(word_embeddings)
        
        # On enlève la diagonale (sim = 1)
        n = len(top_words)
        if n > 1:
            sims = (sim_matrix.sum() - n) / (n*(n-1))  # moyenne des cosinus
            scores.append(sims)

    return np.mean(scores)

# Exemple avec un modèle SentenceTransformer
embedder = SentenceTransformer('all-MiniLM-L6-v2')

for name, model in models.items():
    score = embedding_coherence(model, embedder, top_n=10)
    print(f"{name} - Embedding-based coherence: {score:.4f}")

français - Embedding-based coherence: 0.3487
multilingue - Embedding-based coherence: 0.4008
multilingue_2 - Embedding-based coherence: 0.4078


### Stockage

In [14]:
os.makedirs(Path("../models/topic_modeling"), exist_ok=True)
os.makedirs(Path("../models/topic_modeling/bertopic"), exist_ok=True)
os.makedirs(Path("../models/topic_modeling/bertopic/models"), exist_ok=True)

# Enregistrer un modèle
for name, model in models.items():
    with open(f"../models/topic_modeling/bertopic/models/bertopic_{name}_phrases.pkl", "wb") as f:
        pickle.dump(model, f)

In [19]:
# Enregistrer les topics obtenus
os.makedirs(Path("../models/topic_modeling/bertopic/output"), exist_ok=True)

df["topics"] = models["multilingue_2"].topics_
df.to_csv("../models/topic_modeling/bertopic/output/reviews_phrases_with_topics.csv", index=False, encoding="utf8")

### Visualisation

In [42]:
# CHARGER LES MODELES ENREGISTRES
models = {}
for name in ["français", "multilingue", "multilingue_2"]:
    with open(os.path.join("../models/topic_modeling/bertopic/models", f"bertopic_{name}_phrases.pkl"), 'rb') as f:
        models[name] = pickle.load(f)

In [43]:
all_topics_words = {}
topic_model = models["multilingue_2"]

for topic_id in topic_model.get_topics().keys():
    if topic_id == -1:
        continue  # ignorer les outliers
    all_topics_words[topic_id] = [word for word, _ in topic_model.get_topic(topic_id)]

all_topics_words

{1: ['long',
  'livraison peu',
  'délai livraison',
  'trop long',
  'peu long',
  'long délai',
  'livraison peu long',
  'long délai livraison',
  'livraison trop',
  'délai',
  'peu',
  'livraison trop long',
  'long livraison',
  'longue',
  'trop'],
 2: ['conforme',
  'description',
  'conforme description',
  'attentes',
  'conformes',
  'livraison rapide',
  'description livraison',
  'rapide',
  'livraison délais',
  'conforme description livraison',
  'produit conforme',
  'produits conformes',
  'attentes livraison',
  'conformes attentes',
  'conformes commande'],
 3: ['bonne qualité',
  'bonne',
  'qualité',
  'qualité livraison rapide',
  'qualité livraison',
  'rapide',
  'livraison rapide',
  'produit bonne',
  'produit bonne qualité',
  'rapide produit',
  'bonne qualité livraison',
  'très bonne qualité',
  'livraison rapide produit',
  'qualité très',
  'très bonne'],
 4: ['commanderai',
  'commanderai plus',
  'plus jamais',
  'site commanderai',
  'site commanderai

In [44]:
# topics trouvés
models["français"].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,1,23,1_rien redire_redire_parfait rien_rien,"[rien redire, redire, parfait rien, rien, rien...","[parfait rien à redire ., parfait , rien à red..."
1,2,22,2_plus tôt_tôt_tôt prévu_plus tôt prévu,"[plus tôt, tôt, tôt prévu, plus tôt prévu, pré...","[livré plus tôt que prévu ., et livré plus tôt..."
2,3,21,3_voir temps_temps voir_voir_temps voir temps,"[voir temps, temps voir, voir, temps voir temp...","[a voir dans le temps ., á voir dans le temps ..."
3,4,21,4_rapport qualité prix_rapport qualité_qualité...,"[rapport qualité prix, rapport qualité, qualit...","[bon rapport qualité /prix, bon rapport qualit..."
4,5,18,5_différence prix_expliquer différence prix_ex...,"[différence prix, expliquer différence prix, e...",[pourriez-vous m'expliquer la différence de pr...
...,...,...,...,...,...
192,193,241,193_cela fait_cela_jamais_depuis,"[cela fait, cela, jamais, depuis, site, sais, ...",NaN
193,194,123,194_reçu_prévue_entre commande_date,"[reçu, prévue, entre commande, date, reçu comm...",NaN
194,195,82,195_attentes_conforme_conforme attentes_corres...,"[attentes, conforme, conforme attentes, corres...",NaN
195,196,260,196_service client_service_client_mail,"[service client, service, client, mail, rembou...",NaN


In [45]:
# topics trouvés
models["multilingue"].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,598,0_euros_50_bon achat_achat,"[euros, 50, bon achat, achat, euro, 99, frais,...",[( ils ont trouvé un autre pigeon ) pour l'ins...
1,1,576,1_pieds_bottes_pointure_pied,"[pieds, bottes, pointure, pied, confortables, ...",[j'ai commandé 2 paires de chaussures pointure...
2,2,530,2_carton_emballage_abîmé_cassé,"[carton, emballage, abîmé, cassé, valise, arri...","[l'un des colis est arrivé abîmé , à différent..."
3,3,495,3_vente privée_privée_vente_privées,"[vente privée, privée, vente, privées, ventes ...","[pas chez la vente privée !, vente privée pour..."
4,4,495,4_robes_pantalon_taille_vêtement,"[robes, pantalon, taille, vêtement, trop, qual...","[robes beaucoup trop grande, 3 robes sur 4 ret..."
...,...,...,...,...,...
517,517,256,517_frais_frais port_port_frais livraison,"[frais, frais port, port, frais livraison, che...",NaN
518,518,153,518_bien_tres_rien dire_rien redire,"[bien, tres, rien dire, rien redire, redire, b...",NaN
519,519,73,519_leurs_leurs clients_savent_répondent,"[leurs, leurs clients, savent, répondent, veul...",NaN
520,520,279,520_remboursement_jours_retour_sous,"[remboursement, jours, retour, sous, paiement,...",NaN


In [46]:
# enregistrer les topics trouvés
models["multilingue_2"].get_topic_info().to_csv("../models/topic_modeling/bertopic/output/topics_top_words_phrases.csv", index=False, encoding="utf8")
models["multilingue_2"].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,1,155,1_long_livraison peu_délai livraison_trop long,"[long, livraison peu, délai livraison, trop lo...","[délai de livraison un peu long !, délai livra..."
1,2,133,2_conforme_description_conforme description_at...,"[conforme, description, conforme description, ...",[livraison rapide et produit conforme à mes at...
2,3,108,3_bonne qualité_bonne_qualité_qualité livraiso...,"[bonne qualité, bonne, qualité, qualité livrai...","[sac de bonne qualité, de la bonne qualité, vê..."
3,4,103,4_commanderai_commanderai plus_plus jamais_sit...,"[commanderai, commanderai plus, plus jamais, s...","[je ne commanderai plus jamais, je ne commande..."
4,5,84,5_recommande_site recommande_recommande tout_r...,"[recommande, site recommande, recommande tout,...","[je ne recommande pas ..., je ne recommande pa..."
...,...,...,...,...,...
343,344,305,344_commandé_depuis_vente privée_vente,"[commandé, depuis, vente privée, vente, privée...",NaN
344,345,8,345_avis plus depuis_ben reponse durant_ben re...,"[avis plus depuis, ben reponse durant, ben rep...",NaN
345,346,143,346_satisfaite_articles_intéressants_prix,"[satisfaite, articles, intéressants, prix, trè...",NaN
346,347,2,347_12 quelqu_12 quelqu peu_après certain_1er ...,"[12 quelqu, 12 quelqu peu, après certain, 1er ...",NaN


In [47]:
# Mots-clés associés à un topic
models["français"].get_topic(1)

[('rien redire', np.float64(0.14898756167987542)),
 ('redire', np.float64(0.14858580231951726)),
 ('parfait rien', np.float64(0.13472257784363512)),
 ('rien', np.float64(0.12367654816632775)),
 ('rien dire', np.float64(0.10702398949514447)),
 ('dire', np.float64(0.09709557260353381)),
 ('parfait rien redire', np.float64(0.07998049139630625)),
 ('parfait', np.float64(0.07707143487146415)),
 ('dire rien', np.float64(0.07454389717485463)),
 ('top rien', np.float64(0.07081609375638119)),
 ('rien plus dire', np.float64(0.06337516637036113)),
 ('signaler rien', np.float64(0.057810725735715586)),
 ('plus dire', np.float64(0.057810725735715586)),
 ('dire parfait rien', np.float64(0.05601991093192807)),
 ('dire livraison rapide', np.float64(0.054556983353179184))]

In [48]:
# Mots-clés associés à un topic
models["multilingue"].get_topic(1)

[('pieds', np.float64(0.00749501328256619)),
 ('bottes', np.float64(0.007007370109666849)),
 ('pointure', np.float64(0.006202424817641478)),
 ('pied', np.float64(0.005960175778525571)),
 ('confortables', np.float64(0.005038350557164138)),
 ('commandé', np.float64(0.004311882847860418)),
 ('taille', np.float64(0.003951457166839396)),
 ('37', np.float64(0.0035678827076082286)),
 ('semelle', np.float64(0.0035513392652961225)),
 ('elles', np.float64(0.00337688731285498)),
 ('39', np.float64(0.0033354678713124883)),
 ('deux', np.float64(0.003285620221767201)),
 ('cuir', np.float64(0.0031833465430505525)),
 ('belles', np.float64(0.0030866296786440928)),
 ('commandées', np.float64(0.0030816029618622276))]

In [51]:
fig1 = models["français"].visualize_topics(top_n_topics=10)
fig2 = models["multilingue_2"].visualize_topics(top_n_topics=10)

combined_fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Modèle 1 : Français", "Modèle 2 : Multilingue")
)

for trace in fig1['data']:
    combined_fig.add_trace(trace, row=1, col=1)

for trace in fig2['data']:
    combined_fig.add_trace(trace, row=1, col=2)

combined_fig.update_layout(
    title_text="Répartition des topics",
    showlegend=False,
    height=600,
    width=1000
)

combined_fig.show()

IndexError: index 16 is out of bounds for axis 0 with size 13

In [50]:
models["français"].visualize_barchart(top_n_topics=10)

In [13]:
models["multilingue_2"].visualize_barchart(top_n_topics=10)

In [14]:
models["français"].visualize_hierarchy()

In [15]:
models["multilingue_2"].visualize_hierarchy()